# SoNNeT: pinned upstream Kaggle reproduction

This MoNuSAC notebook checks out `QuIIL/Sonnet` at commit `6cc6c2bbada1084edc82d041d13c307a109806bc`. It executes the authors' model, losses, training, inference, ordinal post-processing, and metrics without translating them. Only the dataset paths in `config.py` are configured, as required by the upstream README. The test-only Ambiguous class is excluded from both ground truth and predictions before scoring.

Enable Kaggle Internet and select a **P100 GPU** when available. The exact upstream runtime is Python 3.6 + TensorFlow 1.12. T4/L4 hardware belongs to a newer CUDA generation and should not be used to claim an exact TF 1.12 GPU reproduction.

In [ ]:
# Edit this cell to match the attached Kaggle dataset.
# Patches must be the upstream 540x540_76x76 .npy format [RGB, inst, type].
from pathlib import Path
import json

SETTINGS = {
    'dataset': 'monusac',
    'input_root': '/kaggle/input/sonnet-monusac',
    'train_patches': 'Train/540x540_76x76',
    'valid_patches': 'Valid/540x540_76x76',
    'test_images': 'Test/Images',
    'test_labels': 'Test/Labels',
    'test_image_extension': '.png',
    'encoder_weights': 'ImageNet_pretrained_EfficientB0.npz',
    'ambiguous_type_ids': [-1, 5],
    'require_ambiguous_mask': True,
    'gpu_id': '0',
    'use_legacy_gpu_package': True,
    'run_train': True,
    'run_inference': True,
    'run_metrics': True,
}
assert SETTINGS['dataset'] == 'monusac'
Path('/kaggle/working/sonnet_settings.json').write_text(json.dumps(SETTINGS, indent=2))
SETTINGS

In [ ]:
# Record hardware and validate all inputs before installing anything.
import json, subprocess
from pathlib import Path
cfg = json.loads(Path('/kaggle/working/sonnet_settings.json').read_text())
root = Path(cfg['input_root'])
subprocess.run(['nvidia-smi'], check=False)
required = ('train_patches', 'valid_patches', 'test_images', 'test_labels', 'encoder_weights')
missing = [str(root / cfg[key]) for key in required if not (root / cfg[key]).exists()]
if missing:
    raise FileNotFoundError('Missing Kaggle inputs:\n' + '\n'.join(missing))
print('train patches:', len(list((root / cfg['train_patches']).glob('*.npy'))))
print('valid patches:', len(list((root / cfg['valid_patches']).glob('*.npy'))))
print('test images:', len(list((root / cfg['test_images']).glob('*' + cfg['test_image_extension']))))
print('test labels:', len(list((root / cfg['test_labels']).glob('*.mat'))))

In [ ]:
# Install an isolated Python 3.6 environment under /kaggle/working.
# This does not modify Kaggle's kernel environment.
import json, os, subprocess
from pathlib import Path
work = Path('/kaggle/working')
cfg = json.loads((work / 'sonnet_settings.json').read_text())
repo = work / 'Sonnet'
env_dir = work / 'sonnet-py36'
commit = '6cc6c2bbada1084edc82d041d13c307a109806bc'
if not (repo / '.git').exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/QuIIL/Sonnet.git', str(repo)])
subprocess.check_call(['git', '-C', str(repo), 'fetch', '--all', '--tags'])
subprocess.check_call(['git', '-C', str(repo), 'checkout', '--detach', commit])
assert subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip() == commit
if not (env_dir / 'bin/python').exists():
    installer = work / 'miniconda-py36.sh'
    subprocess.check_call(['wget', '-q', 'https://repo.anaconda.com/miniconda/Miniconda3-py36_4.9.2-Linux-x86_64.sh', '-O', str(installer)])
    subprocess.check_call(['bash', str(installer), '-b', '-p', str(env_dir)])
pip = str(env_dir / 'bin/pip')
subprocess.check_call([pip, 'install', '--upgrade', 'pip<22', 'setuptools<60', 'wheel'])
packages = [
    'numpy==1.19.2', 'scipy==1.1.0', 'pandas==1.1.0', 'matplotlib==3.3.4',
    'scikit_image==0.15.0', 'scikit_learn==0.24.2',
    'opencv_python_headless==3.4.8.29', 'tensorpack==0.9.0.1',
    ('tensorflow-gpu==1.12.0' if cfg['use_legacy_gpu_package'] else 'tensorflow==1.12.0'),
]
subprocess.check_call([pip, 'install', *packages])

In [ ]:
# Verify the legacy runtime before training. If GPU visibility is False,
# switch to a P100 runtime or set use_legacy_gpu_package=False for CPU.
import subprocess
python36 = '/kaggle/working/sonnet-py36/bin/python'
code = "import tensorflow as tf; print('TensorFlow', tf.__version__); print('GPU visible', tf.test.is_gpu_available(cuda_only=True)); assert tf.__version__ == '1.12.0'"
subprocess.check_call([python36, '-c', code])

In [ ]:
# Configure only fields the upstream README tells users to set.
import json, shutil, subprocess
from pathlib import Path
work = Path('/kaggle/working')
repo = work / 'Sonnet'
cfg = json.loads((work / 'sonnet_settings.json').read_text())
root = Path(cfg['input_root'])
text = (repo / 'config.py').read_text()
replacements = {
    "self.data_type = 'consep'": f"self.data_type = {cfg['dataset']!r}",
    'self.nr_types = 5': f"self.nr_types = {4 if cfg['dataset'] == 'glysac' else 5}",
    "self.train_dir = ['/media/tandoan/data2/CoNSeP/Train/%s/'  % data_code_dict[self.model_type]]": f"self.train_dir = [{str(root / cfg['train_patches'])!r}]",
    "self.valid_dir = ['/home/tandoan/work/PanNuke/Valid/%s' % data_code_dict[self.model_type]]": f"self.valid_dir = [{str(root / cfg['valid_patches'])!r}]",
    "self.log_path = '/media/tandoan/data2/logs/logs_test'": "self.log_path = '/kaggle/working/sonnet-logs'",
    "self.inf_auto_find_chkpt = False": "self.inf_auto_find_chkpt = True",
    "self.inf_imgs_ext = '.png'": f"self.inf_imgs_ext = {cfg['test_image_extension']!r}",
    "self.inf_data_dir = '/media/tandoan/data2/CoNSeP/Test/Images'": f"self.inf_data_dir = {str(root / cfg['test_images'])!r}",
    "self.inf_output_dir = 'output/test/'": "self.inf_output_dir = '/kaggle/working/sonnet-predictions/'",
}
for old, new in replacements.items():
    if text.count(old) != 1:
        raise RuntimeError('Pinned config no longer matches: ' + old)
    text = text.replace(old, new)
(repo / 'config.py').write_text(text)
shutil.copy2(root / cfg['encoder_weights'], repo / 'ImageNet_pretrained_EfficientB0.npz')
print(subprocess.check_output(['git', '-C', str(repo), 'diff', '--', 'config.py'], text=True))

In [ ]:
# Authors' unchanged three-phase training schedule: 50 + 25 + 25 epochs.
import json, subprocess
from pathlib import Path
cfg = json.loads(Path('/kaggle/working/sonnet_settings.json').read_text())
if cfg['run_train']:
    subprocess.check_call(['/kaggle/working/sonnet-py36/bin/python', 'train.py', '--gpu=' + cfg['gpu_id']], cwd='/kaggle/working/Sonnet')

In [ ]:
# Authors' unchanged inference and ordinal post-processing.
import json, subprocess
from pathlib import Path
cfg = json.loads(Path('/kaggle/working/sonnet_settings.json').read_text())
python36 = '/kaggle/working/sonnet-py36/bin/python'
if cfg['run_inference']:
    subprocess.check_call([python36, 'infer.py', '--gpu=' + cfg['gpu_id']], cwd='/kaggle/working/Sonnet')
    subprocess.check_call([python36, 'process.py'], cwd='/kaggle/working/Sonnet')

In [ ]:
# Build scored copies using the official MoNuSAC ignore-region principle.
# Uploaded ground truth and raw SoNNeT predictions are never overwritten.
import json
from pathlib import Path
import numpy as np
from scipy.io import loadmat, savemat

cfg = json.loads(Path('/kaggle/working/sonnet_settings.json').read_text())
truth_source = Path(cfg['input_root']) / cfg['test_labels']
pred_source = Path('/kaggle/working/sonnet-predictions/_proc')
truth_scored = Path('/kaggle/working/sonnet-scored-test-labels')
pred_scored = Path('/kaggle/working/sonnet-scored-predictions')
ignore_dir = Path('/kaggle/working/monusac-ignore-masks')
for directory in (truth_scored, pred_scored, ignore_dir):
    directory.mkdir(parents=True, exist_ok=True)

def instance_types(data, instance_map):
    if 'inst_type' in data:
        return np.asarray(data['inst_type']).reshape(-1).astype(np.int32)
    pixel_types = np.asarray(data['type_map']).astype(np.int32)
    result = []
    for instance_id in range(1, int(instance_map.max()) + 1):
        values = pixel_types[instance_map == instance_id]
        values = values[values != 0]
        if len(values):
            labels, counts = np.unique(values, return_counts=True)
            result.append(int(labels[np.argmax(counts)]))
        else:
            result.append(0)
    return np.asarray(result, dtype=np.int32)

def rebuild_annotation(data, ignore):
    original = np.asarray(data['inst_map']).astype(np.int32)
    old_types = instance_types(data, original)
    masked = original.copy()
    masked[ignore] = 0
    remapped = np.zeros_like(masked)
    new_types, centroids = [], []
    for new_id, old_id in enumerate(np.unique(masked)[1:], start=1):
        mask = masked == old_id
        remapped[mask] = new_id
        new_types.append(int(old_types[old_id - 1]) if old_id - 1 < len(old_types) else 0)
        yy, xx = np.nonzero(mask)
        centroids.append([float(xx.mean()), float(yy.mean())])
    pixel_types = np.asarray(data.get('type_map', np.zeros_like(original))).astype(np.int32)
    pixel_types = pixel_types.copy()
    pixel_types[ignore] = 0
    return {
        'inst_map': remapped,
        'type_map': pixel_types,
        'inst_type': np.asarray(new_types, dtype=np.int32)[:, None],
        'inst_centroid': np.asarray(centroids, dtype=np.float32).reshape(-1, 2),
    }

ignored_pixels = 0
for truth_path in sorted(truth_source.glob('*.mat')):
    data = loadmat(truth_path)
    inst_map = np.asarray(data['inst_map']).astype(np.int32)
    if 'ignore_map' in data:
        ignore = np.asarray(data['ignore_map']).astype(bool)
    else:
        ignore = np.zeros(inst_map.shape, dtype=bool)
        if 'type_map' in data:
            ignore |= np.isin(np.asarray(data['type_map']), cfg['ambiguous_type_ids'])
        old_types = instance_types(data, inst_map)
        ignored_ids = np.flatnonzero(np.isin(old_types, cfg['ambiguous_type_ids'])) + 1
        ignore |= np.isin(inst_map, ignored_ids)
    ignored_pixels += int(ignore.sum())
    np.save(ignore_dir / (truth_path.stem + '.npy'), ignore)
    savemat(truth_scored / truth_path.name, rebuild_annotation(data, ignore))
if cfg['require_ambiguous_mask'] and ignored_pixels == 0:
    raise RuntimeError('No Ambiguous pixels were found. Supply type ID 5/-1 or an explicit ignore_map; otherwise predictions there would be false positives.')

for pred_path in sorted(pred_source.glob('*.mat')):
    ignore_path = ignore_dir / (pred_path.stem + '.npy')
    if not ignore_path.exists():
        raise FileNotFoundError('Missing test ignore mask for ' + pred_path.name)
    savemat(pred_scored / pred_path.name, rebuild_annotation(loadmat(pred_path), np.load(ignore_path)))
print('Excluded Ambiguous pixels:', ignored_pixels)
print('Scored GT:', truth_scored)
print('Scored predictions:', pred_scored)

In [ ]:
# Authors' instance and type metric commands.
import json, subprocess
from pathlib import Path
cfg = json.loads(Path('/kaggle/working/sonnet_settings.json').read_text())
python36 = '/kaggle/working/sonnet-py36/bin/python'
pred = '/kaggle/working/sonnet-scored-predictions/'
truth = '/kaggle/working/sonnet-scored-test-labels/'
if cfg['run_metrics']:
    subprocess.check_call([python36, 'compute_stats.py', '--mode=instance', '--pred_dir=' + pred, '--true_dir=' + truth], cwd='/kaggle/working/Sonnet')
    subprocess.check_call([python36, 'compute_stats.py', '--mode=type', '--pred_dir=' + pred, '--true_dir=' + truth], cwd='/kaggle/working/Sonnet')

In [ ]:
# Save an audit manifest and bundle Kaggle outputs for download.
import json, shutil, subprocess
from pathlib import Path
work = Path('/kaggle/working')
repo = work / 'Sonnet'
bundle = work / 'sonnet-kaggle-artifacts'
bundle.mkdir(exist_ok=True)
manifest = {
    'repository': 'https://github.com/QuIIL/Sonnet',
    'commit': subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip(),
    'source_diff': subprocess.check_output(['git', '-C', str(repo), 'diff'], text=True),
    'settings': json.loads((work / 'sonnet_settings.json').read_text()),
}
(bundle / 'reproduction_manifest.json').write_text(json.dumps(manifest, indent=2))
for name in ('sonnet-logs', 'sonnet-predictions', 'sonnet-scored-test-labels', 'sonnet-scored-predictions', 'monusac-ignore-masks'):
    source = work / name
    if source.exists():
        shutil.copytree(source, bundle / name, dirs_exist_ok=True)
archive = shutil.make_archive(str(work / 'sonnet-kaggle-artifacts'), 'zip', bundle)
print('Download from Kaggle output:', archive)